# 17 — Sync vLLM CrafText rollout

Перед запуском на GPU-сервере подготовьте окружение и локальный snapshot:

```bash
uv sync --extra envs --extra prompts --extra vllm --extra examples
# model path берётся из configs/inference/vllm/qwen25_05b_sync.yaml
```

Notebook ничего не скачивает неявно. Если vLLM или snapshot отсутствуют, preflight-ячейка остановится с понятной ошибкой.

Этот notebook доказывает минимальный sync путь: strict generation YAML → CrafText task → MegaPrompts → Qwen chat template → vLLM batch generation → strict action decode/fallback → batched CrafText step → replay artifacts.


In [ ]:
import os

# Must run before importing JAX/vLLM/Tunix. Restart the kernel after changing these.
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'

print('XLA_PYTHON_CLIENT_PREALLOCATE=', os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'])
print('VLLM_WORKER_MULTIPROC_METHOD=', os.environ['VLLM_WORKER_MULTIPROC_METHOD'])


In [ ]:
from pathlib import Path

from tunix_craftext.inference import load_generation_pipeline_config

ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'pyproject.toml').is_file()),
    None,
)
if ROOT is None:
    raise RuntimeError('Run this notebook from inside the tunix-craftext repository')

GENERATION_CONFIG = ROOT / 'configs/inference/vllm/qwen25_05b_sync.yaml'
generation = load_generation_pipeline_config(GENERATION_CONFIG)
SNAPSHOT = Path(generation.profile.model)
if not SNAPSHOT.is_absolute():
    SNAPSHOT = ROOT / SNAPSHOT
if not SNAPSHOT.is_dir():
    raise FileNotFoundError(f'Local model snapshot is missing: {SNAPSHOT}')

CONFIG_PATH = ROOT / 'configs/env/text/qwen_craftext.yaml'
OUTPUT_DIR = ROOT / 'artifacts/trajectories/vllm-sync-rollout'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('repo:', ROOT)
print('generation config:', GENERATION_CONFIG)
print('snapshot:', SNAPSHOT)


In [ ]:
from tunix_craftext.env.config import load_mvp_config
from tunix_craftext.env.runtime import build_craftext_runtime
from tunix_craftext.env.tasks import CrafTextTaskSampler

config = load_mvp_config(CONFIG_PATH)
runtime = build_craftext_runtime(config)
task_sampler = CrafTextTaskSampler.from_runtime(
    runtime,
    horizon=config.environment.horizon,
    mode='fixed',
    fixed_instruction_index=config.environment.instruction_index,
    goal_prefix=config.run.goal,
)
prepared_goal, prepared_instruction_index = task_sampler.task_at(config.environment.instruction_index)
print('instruction_index:', prepared_instruction_index)
print('goal:', prepared_goal)


In [ ]:
from transformers import AutoTokenizer

from tunix_craftext.env.prompts import MegaPromptRenderer, RenderedPrompt

base_renderer = MegaPromptRenderer(config.prompt.template)
tokenizer = AutoTokenizer.from_pretrained(str(SNAPSHOT), trust_remote_code=True)

class QwenChatTemplateRenderer:
    def __init__(self, renderer, tokenizer):
        self.renderer = renderer
        self.tokenizer = tokenizer

    def render(self, context):
        rendered = self.renderer.render(context)
        chat_text = self.tokenizer.apply_chat_template(
            [
                {
                    'role': 'system',
                    'content': 'You are a CrafText agent. Return exactly one <action>LABEL</action>.',
                },
                {'role': 'user', 'content': rendered.text},
            ],
            add_generation_prompt=True,
            tokenize=False,
        )
        if not isinstance(chat_text, str) or not chat_text.strip():
            raise ValueError('Qwen chat template did not return text')
        return RenderedPrompt(chat_text, rendered.actions, rendered.template_name)

renderer = QwenChatTemplateRenderer(base_renderer, tokenizer)


In [ ]:
from dataclasses import replace

from tunix_craftext.inference import RequestsLlmBackend, VllmInferenceEngine

engine_profile = replace(
    generation.profile,
    model=str(SNAPSHOT),
    metadata={**dict(generation.profile.metadata), 'notebook': '17_sync_vllm_craftext_rollout.ipynb'},
)
engine = VllmInferenceEngine.from_profile(engine_profile)
backend = RequestsLlmBackend(engine)
print('engine:', engine.profile)
print('tunix rollout kwargs:', generation.tunix.to_tunix_rollout_kwargs())


In [ ]:
from tunix_craftext.env.prompts import RenderedPrompt
from tunix_craftext.inference import GenerationBatch, collect_generation_results_sync
from tunix_craftext.models.llm import LlmRequest

sample_prompt = RenderedPrompt(
    '''<|im_start|>user
Return <action>NOOP</action><|im_end|>
<|im_start|>assistant
''',
    runtime.actions,
    'contract-smoke',
)
sync_records = collect_generation_results_sync(
    engine,
    (
        GenerationBatch(
            (LlmRequest(sample_prompt, max_new_tokens=min(4, generation.tunix.max_tokens_to_generate)),),
            group_id='sync-smoke',
            policy_version=engine.profile.policy_version,
        ),
    ),
)
print('sync collector smoke:', sync_records[0].result.responses[0].raw_text)


In [ ]:
from tunix_craftext.rollouts.batched import HostBatchPolicy, collect_batched_text_rollout_profiled, cpu_environment_device_policy

profiled_rollout = collect_batched_text_rollout_profiled(
    runtime.adapter,
    renderer,
    backend,
    actions=runtime.actions,
    batch_size=2,
    horizon=4,
    seed=config.run.seed,
    goal=prepared_goal,
    max_new_tokens=8,
    invalid_action='fallback',
    fallback_action_id=runtime.actions.index_of('NOOP'),
    host_batch_policy=HostBatchPolicy(prompt_workers=4),
    device_policy=cpu_environment_device_policy(),
)

print('phase totals ms:', profiled_rollout.phase_totals_ms())
print('event totals:', profiled_rollout.event_totals())
for index, timing in enumerate(profiled_rollout.timings):
    print(
        'timing step',
        index,
        {
            'env_step_ms': timing.decision.environment_step_ms,
            'reset_ms': timing.reset_ms,
            'replace_finished_ms': timing.replace_finished_ms,
            'finished_count': timing.finished_count,
            'reset_invoked': timing.reset_invoked,
            'total_ms': timing.total_ms,
        },
    )
rollout = profiled_rollout.rollout

for step, decision in enumerate(rollout.decisions):
    print('step', step)
    print('  actions:', [action.label for action in decision.actions])
    print('  rewards:', decision.transition.reward.tolist())
    print('  fallbacks:', decision.fallback_used.tolist())


In [ ]:
from tunix_craftext.artifacts.replay import save_replay
from tunix_craftext.rollouts.batched import replays_from_batched_rollout

replays = replays_from_batched_rollout(
    rollout,
    config_path=str(CONFIG_PATH.relative_to(ROOT)),
    commit='notebook-sync-vllm',
    backend=f'{engine.profile.backend}:{engine.profile.model}',
)
for env_index, replay in enumerate(replays):
    path = OUTPUT_DIR / f'env-{env_index}.json'
    save_replay(path, replay)
    print('saved', path, 'steps=', len(replay.steps))
